In [2]:
# ==========================================================
# Conversion Modeling
# Experiment 1-3:
# 1. Original Logistic Regression
# 2. Balanced Logistic Regression
# 3. Balanced Logistic + log(total ads)
# ==========================================================

# =========================
# 1. Import Libraries
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)


# =========================
# 2. Load Dataset
# =========================

df = pd.read_csv("D:\\Python projects\\AB test\\marketing_AB.csv")

if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# =========================
# 3. Feature Preparation
# =========================

features = [
    "test group",
    "total ads",
    "most ads day",
    "most ads hour"
]

X = df[features]
y = df["converted"]


# =========================
# 4. Train Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================
# 5. Preprocessing
# =========================

categorical_features = [
    "test group",
    "most ads day",
    "most ads hour"
]

numeric_features = [
    "total ads"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(drop="first"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)


# =========================
# 6. Model Evaluation Function
# =========================

def evaluate_model(model, X_test, y_test, model_name):

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

    print("\n==============================")
    print(model_name)
    print("==============================")

    for key, value in result.items():
        if key != "Model":
            print(f"{key}: {value:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return result


# ==========================================================
# Experiment 1
# Original Logistic Regression
# ==========================================================

baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

baseline_model.fit(
    X_train,
    y_train
)

baseline_result = evaluate_model(
    baseline_model,
    X_test,
    y_test,
    "Original Logistic Regression"
)


# ==========================================================
# Experiment 2
# Balanced Logistic Regression
# ==========================================================

balanced_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

balanced_model.fit(
    X_train,
    y_train
)

balanced_result = evaluate_model(
    balanced_model,
    X_test,
    y_test,
    "Balanced Logistic Regression"
)
# ==========================================================
# Experiment 3
# Balanced Logistic Regression + log(total ads)
# ==========================================================

df["log_ads"] = np.log1p(df["total ads"])

features_log = [
    "test group",
    "log_ads",
    "most ads day",
    "most ads hour"
]

X_log = df[features_log]

# Use the same train/test index as previous experiments

X_train_log = X_log.loc[X_train.index]
X_test_log = X_log.loc[X_test.index]

y_train_log = y.loc[y_train.index]
y_test_log = y.loc[y_test.index]


categorical_features_log = [
    "test group",
    "most ads day",
    "most ads hour"
]

numeric_features_log = [
    "log_ads"
]


preprocessor_log = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(drop="first"),
            categorical_features_log
        ),
        (
            "num",
            "passthrough",
            numeric_features_log
        )
    ]
)


log_ads_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_log
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)


log_ads_model.fit(
    X_train_log,
    y_train_log
)


log_ads_result = evaluate_model(
    log_ads_model,
    X_test_log,
    y_test_log,
    "Balanced Logistic + log ads"
)

# ==========================================================
# Experiment 4
# Balanced Logistic + Interaction Feature
# ==========================================================

# Create interaction feature
df["ad_ads_interaction"] = (
    (df["test group"] == "ad").astype(int)
    * df["total ads"]
)

features_interaction = [
    "test group",
    "total ads",
    "ad_ads_interaction",
    "most ads day",
    "most ads hour"
]

X_interaction = df[features_interaction]
y_interaction = df["converted"]


# Use the same train/test split

X_train_interaction = X_interaction.loc[X_train.index]
X_test_interaction = X_interaction.loc[X_test.index]

y_train_interaction = y.loc[y_train.index]
y_test_interaction = y.loc[y_test.index]


# Preprocessing

categorical_features_interaction = [
    "test group",
    "most ads day",
    "most ads hour"
]

numeric_features_interaction = [
    "total ads",
    "ad_ads_interaction"
]


preprocessor_interaction = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(drop="first"),
            categorical_features_interaction
        ),
        (
            "num",
            "passthrough",
            numeric_features_interaction
        )
    ]
)


# Build model

interaction_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_interaction
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

# Train

interaction_model.fit(
    X_train_interaction,
    y_train_interaction
)

# Evaluate

interaction_result = evaluate_model(
    interaction_model,
    X_test_interaction,
    y_test_interaction,
    "Balanced Logistic + interaction"
)

# ==========================================================
# Final Model Comparison
# ==========================================================
comparison = pd.DataFrame(
    [
        baseline_result,
        balanced_result,
        log_ads_result,
        interaction_result
    ],
    index=[
        1,
        2,
        3,
        4
    ]
)

print("\n==============================")
print("Final Model Comparison")
print("==============================")

print(
    comparison.round(4)
)
# ==========================================================
# Model Interpretation
# Logistic Regression Coefficient Analysis
# ==========================================================

# Get feature names after preprocessing
feature_names = (
    interaction_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get model coefficients

coefficients = (
    interaction_model
    .named_steps["classifier"]
    .coef_[0]
)

# Create coefficient dataframe

coef_df = pd.DataFrame(
    {
        "Feature": feature_names,
        "Coefficient": coefficients
    }
)

# Calculate odds ratio

coef_df["Odds_Ratio"] = np.exp(
    coef_df["Coefficient"]
)

# Sort by impact

coef_df = (
    coef_df
    .sort_values(
        "Odds_Ratio",
        ascending=False
    )
    .reset_index(drop=True)
)

print("==============================")
print("Feature Impact Analysis")
print("==============================")

print(
    coef_df.round(4)
)
# ==========================================================
# Experiment 5
# Threshold Optimization
# Based on Balanced Logistic + interaction
# ==========================================================

# Get prediction probability

y_prob = interaction_model.predict_proba(
    X_test_interaction
)[:, 1]


# Test different classification thresholds

thresholds = np.arange(
    0.1,
    0.6,
    0.05
)

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (
        y_prob >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(
                y_test_interaction,
                y_pred_threshold
            ),
            "Recall": recall_score(
                y_test_interaction,
                y_pred_threshold
            ),
            "F1": f1_score(
                y_test_interaction,
                y_pred_threshold
            )
        }
    )


threshold_table = pd.DataFrame(
    threshold_results
)


print("\n==============================")
print("Threshold Performance")
print("==============================")

print(
    threshold_table.round(4)
)


# Select threshold with highest F1 score

best_threshold = (
    threshold_table
    .sort_values(
        "F1",
        ascending=False
    )
    .iloc[0]
)


print("\nBest Threshold:")
print(best_threshold)


# Evaluate final model using optimal threshold

optimal_threshold = best_threshold["Threshold"]

y_pred_optimal = (
    y_prob >= optimal_threshold
).astype(int)


print("\n==============================")
print("Optimized Threshold Result")
print("==============================")

print(
    "Selected Threshold:",
    optimal_threshold
)

print(
    "\nPrecision:",
    precision_score(
        y_test_interaction,
        y_pred_optimal
    )
)

print(
    "Recall:",
    recall_score(
        y_test_interaction,
        y_pred_optimal
    )
)

print(
    "F1:",
    f1_score(
        y_test_interaction,
        y_pred_optimal
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_interaction,
        y_pred_optimal
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_interaction,
        y_pred_optimal
    )
)
# ==========================================================
# Experiment 5.1
# Recall-oriented Threshold Selection
# ==========================================================

# Prediction probability

y_prob = interaction_model.predict_proba(
    X_test_interaction
)[:, 1]


thresholds = np.arange(
    0.1,
    0.6,
    0.01
)

recall_results = []


for threshold in thresholds:

    y_pred_threshold = (
        y_prob >= threshold
    ).astype(int)

    recall_results.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(
                y_test_interaction,
                y_pred_threshold
            ),
            "Recall": recall_score(
                y_test_interaction,
                y_pred_threshold
            ),
            "F1": f1_score(
                y_test_interaction,
                y_pred_threshold
            )
        }
    )


recall_table = pd.DataFrame(
    recall_results
)


# Set recall requirement

target_recall = 0.80


candidate_thresholds = (
    recall_table[
        recall_table["Recall"] >= target_recall
    ]
)


best_recall_threshold = (
    candidate_thresholds
    .sort_values(
        "Precision",
        ascending=False
    )
    .iloc[0]
)


print("==============================")
print("Recall-oriented Threshold")
print("==============================")

print(
    best_recall_threshold
)


# Evaluate selected threshold

optimal_threshold = (
    best_recall_threshold["Threshold"]
)


y_pred_recall = (
    y_prob >= optimal_threshold
).astype(int)


print("\nSelected Threshold:")
print(optimal_threshold)

print("\nPrecision:")
print(
    precision_score(
        y_test_interaction,
        y_pred_recall
    )
)

print("\nRecall:")
print(
    recall_score(
        y_test_interaction,
        y_pred_recall
    )
)

print("\nF1:")
print(
    f1_score(
        y_test_interaction,
        y_pred_recall
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_interaction,
        y_pred_recall
    )
)
# ==========================================================
# Experiment 5.2
# Profit-oriented Threshold Selection
# ==========================================================

revenue_per_conversion = 10

cost_per_target = 0.1
profit_results = []

for threshold in thresholds:

    y_pred_threshold = (
        y_prob >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_test_interaction,
        y_pred_threshold
    )

    tn, fp, fn, tp = cm.ravel()
    targeted_users = tp + fp

    revenue = (
        tp *
        revenue_per_conversion
    )
    cost = (
        targeted_users *
        cost_per_target
    )

    profit = revenue - cost

    profit_results.append(
        {
            "Threshold": threshold,
            "TP": tp,
            "FP": fp,
            "Targeted Users": targeted_users,
            "Revenue": revenue,
            "Cost": cost,
            "Profit": profit
        }
    )

profit_table = pd.DataFrame(
    profit_results
)

best_profit_threshold = (
    profit_table
    .sort_values(
        "Profit",
        ascending=False
    )
    .iloc[0]
)

print("==============================")
print("Profit-oriented Threshold")
print("==============================")

print(
    best_profit_threshold
)


Original Logistic Regression
Accuracy: 0.9736
ROC-AUC: 0.8142
PR-AUC: 0.1259
Precision: 0.1974
Recall: 0.0152
F1: 0.0282

Confusion Matrix:
[[114469    183]
 [  2924     45]]

Classification Report:
              precision    recall  f1-score   support

       False       0.98      1.00      0.99    114652
        True       0.20      0.02      0.03      2969

    accuracy                           0.97    117621
   macro avg       0.59      0.51      0.51    117621
weighted avg       0.96      0.97      0.96    117621


Balanced Logistic Regression
Accuracy: 0.8574
ROC-AUC: 0.8545
PR-AUC: 0.1349
Precision: 0.1177
Recall: 0.7154
F1: 0.2021

Confusion Matrix:
[[98730 15922]
 [  845  2124]]

Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.86      0.92    114652
        True       0.12      0.72      0.20      2969

    accuracy                           0.86    117621
   macro avg       0.55      0.79      0.56    117621
weigh